# 1-Dars

In [862]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [863]:
# Load Dataset
df = pd.read_csv("titanic_class.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [864]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


# Preprocessing

In [865]:
df=df.drop(['PassengerId','Name','Ticket','Cabin'],axis=1)

In [866]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [867]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [868]:
# Missing
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [869]:
# Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

## Train_Test_Split

In [870]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Scaling

In [871]:
# Scaling (faqat linear + SVM) qolgan algorithmlar uchun scaling shart emas
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# StandardScaler -> SVM va Logistic Regression uchun u eng stabil va mathematically to'g'ri variant

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Baseline (predict)

In [872]:
def baseline(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1_score": f1_score(y_true, y_pred)
    }

## Threshold (predict + proba)

In [873]:
def threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred),
        "Recall": recall_score(y_true, pred),
        "F1_score": f1_score(y_true, pred)
    }

## Models without Scaling (Tree-based)

In [874]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
dt_proba = dt.predict_proba(X_test)[:, 1]
# [:, 1] -> hamma samplelar uchun Survived ehtimolligini chiqaradi

In [875]:
# Probability of class
print("Raw probabilities:", dt_proba[0])
dt_positive_prob = dt_proba[0]
print(f"Model confidence for positive class: {dt_positive_prob*100:.2f}%")

Raw probabilities: 0.0
Model confidence for positive class: 0.00%


In [876]:
# Decision Tree result
dt_base = baseline(y_test, dt_pred)

dt_03 = threshold(y_test, dt_proba, 0.3)
dt_05 = threshold(y_test, dt_proba, 0.5)
dt_08 = threshold(y_test, dt_proba, 0.8)

pd.DataFrame([dt_base, dt_03, dt_05, dt_08],index=['Baseline',"0.3","0.5","0.8"])

,Accuracy,Precision,Recall,F1_score
Baseline,0.782123,0.721519,0.770270,0.745098
0.3,0.787709,0.704545,0.837838,0.765432
0.5,0.793296,0.717647,0.824324,0.767296
0.8,0.782123,0.721519,0.770270,0.745098


In [877]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

In [878]:
# Probability of class
print("Raw probabilities:", rf_proba[0])
rf_positive_prob = rf_proba[0]
print(f"Model confidence for positive class: {rf_positive_prob*100:.2f}%")

Raw probabilities: 0.33
Model confidence for positive class: 33.00%


In [879]:
# Random Forest result
rf_base = baseline(y_test, rf_pred)

rf_03 = threshold(y_test, rf_proba, 0.3)
rf_05 = threshold(y_test, rf_proba, 0.5)
rf_08 = threshold(y_test, rf_proba, 0.8)

pd.DataFrame([rf_base, rf_03, rf_05, rf_08], index=['Baseline',"0.3", "0.5", "0.8"])

,Accuracy,Precision,Recall,F1_score
Baseline,0.821229,0.808824,0.743243,0.774648
0.3,0.798883,0.711111,0.864865,0.780488
0.5,0.815642,0.797101,0.743243,0.769231
0.8,0.826816,0.921569,0.635135,0.752000


In [880]:
# XGBoost
from xgboost import XGBClassifier
xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

In [881]:
# Probability of class
print("Raw probabilities:", xgb_proba[0])
xgb_positive_prob = xgb_proba[0]
print(f"Model confidence for positive class: {xgb_positive_prob*100:.2f}%")

Raw probabilities: 0.012775595
Model confidence for positive class: 1.28%


In [882]:
# XGBoost result
xgb_base = baseline(y_test, xgb_pred)

xgb_03 = threshold(y_test, xgb_proba, 0.3)
xgb_05 = threshold(y_test, xgb_proba, 0.5)
xgb_08 = threshold(y_test, xgb_proba, 0.8)

pd.DataFrame([xgb_base, xgb_03, xgb_05, xgb_08],index=['Baseline',"0.3", "0.5", "0.8"]) 

,Accuracy,Precision,Recall,F1_score
Baseline,0.798883,0.756757,0.756757,0.756757
0.3,0.770950,0.689655,0.810811,0.745342
0.5,0.798883,0.756757,0.756757,0.756757
0.8,0.810056,0.857143,0.648649,0.738462


## Models with Scaling(Linear + SVM)

In [883]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

In [884]:
# Probability of class
print("Raw probabilities:", lr_proba[0])
lr_positive_prob = lr_proba[0]
print(f"Model confidence for positive class: {lr_positive_prob*100:.2f}%")

Raw probabilities: 0.1084525514711013
Model confidence for positive class: 10.85%


In [885]:
# Logistic Regression result
lr_base = baseline(y_test, lr_pred)

lr_03 = threshold(y_test, lr_proba, 0.3)
lr_05 = threshold(y_test, lr_proba, 0.5)
lr_07 = threshold(y_test, lr_proba, 0.7) # 0.8 qilsak pre=1 bo'lib recall tushib ketdi

pd.DataFrame([lr_base, lr_03, lr_05, lr_07], index=['Baseline', "0.3", "0.5", "0.7"])

,Accuracy,Precision,Recall,F1_score
Baseline,0.804469,0.782609,0.729730,0.755245
0.3,0.782123,0.688172,0.864865,0.766467
0.5,0.804469,0.782609,0.729730,0.755245
0.7,0.804469,0.933333,0.567568,0.705882


In [886]:
# SVM
from sklearn.svm import SVC
svc = SVC(probability=True)
svc.fit(X_train_scaled, y_train)

svc_pred = svc.predict(X_test_scaled)
svc_proba = svc.predict_proba(X_test_scaled)[:, 1]

In [887]:
# Probability of class
print("Raw probabilities:", svc_proba[0])
svc_positive_prob = svc_proba[0]
print(f"Model confidence for positive class: {svc_positive_prob*100:.2f}%")

Raw probabilities: 0.17139898625840572
Model confidence for positive class: 17.14%


In [888]:
# SVC result
svc_base = baseline(y_test, svc_pred)

svc_03 = threshold(y_test, svc_proba, 0.3)
svc_05 = threshold(y_test, svc_proba, 0.5)
svc_08 = threshold(y_test, svc_proba, 0.8)

pd.DataFrame([svc_base, svc_03, svc_05, svc_08], index=['Baseline', "0.3", "0.5", "0.8"])

,Accuracy,Precision,Recall,F1_score
Baseline,0.815642,0.815385,0.716216,0.762590
0.3,0.782123,0.739726,0.729730,0.734694
0.5,0.815642,0.815385,0.716216,0.762590
0.8,0.798883,0.839286,0.635135,0.723077


## Tabulate

In [889]:
def model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred)
    }

In [890]:
dt_result = model(
    "Decision Tree",
    y_test,
    dt.predict(X_test)
)

In [891]:
rf_result = model(
    "Random Forest",
    y_test,
    rf.predict(X_test)
)

In [892]:
xgb_result = model(
    "XGBoost",
    y_test,
    xgb.predict(X_test)
)

In [893]:
lr_result = model(
    "Logistic Regression",
    y_test,
    lr.predict(X_test_scaled)
)

In [894]:
svc_result = model(
    "SVM",
    y_test,
    svc.predict(X_test_scaled)
)

In [895]:
results_df = pd.DataFrame([
    dt_result,
    rf_result,
    xgb_result,
    lr_result,
    svc_result
])

In [896]:
from tabulate import tabulate
print(tabulate(results_df, headers="keys", tablefmt="github", floatfmt=".2f"))

|    | Model               |   Accuracy |   Precision |   Recall |   F1-score |
|----|---------------------|------------|-------------|----------|------------|
|  0 | Decision Tree       |       0.78 |        0.72 |     0.77 |       0.75 |
|  1 | Random Forest       |       0.82 |        0.81 |     0.74 |       0.77 |
|  2 | XGBoost             |       0.80 |        0.76 |     0.76 |       0.76 |
|  3 | Logistic Regression |       0.80 |        0.78 |     0.73 |       0.76 |
|  4 | SVM                 |       0.82 |        0.82 |     0.72 |       0.76 |


In [897]:
# Eng yaxshi model
best_model = results_df.sort_values(by="F1-score", ascending=False).iloc[0]
    # sort_values(...) -> kattadan kichichka tartiblaydi
    # .iloc[0] -> eng yuqori qatordan (1-chi row) bitta modelni oladi

print("BEST MODEL:")
print(best_model)

BEST MODEL:
Model        Random Forest
Accuracy          0.821229
Precision         0.808824
Recall            0.743243
F1-score          0.774648
Name: 1, dtype: object
